In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load files into DataFrames (cached variables for reuse)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Helper to build compact overview for a dataframe
def build_overview(df, name_col, file_label):
    # safety: ensure timestamp and cmdb_id exist
    ts_min = int(df['timestamp'].min()) if 'timestamp' in df.columns and not df['timestamp'].isna().all() else None
    ts_max = int(df['timestamp'].max()) if 'timestamp' in df.columns and not df['timestamp'].isna().all() else None
    row_count = len(df)
    unique_cmdb = int(df['cmdb_id'].nunique()) if 'cmdb_id' in df.columns else 0
    cmdb_counts = df['cmdb_id'].value_counts().head(20).reset_index().rename(columns={'index':'cmdb_id', 'cmdb_id':'count'}) if 'cmdb_id' in df.columns else pd.DataFrame(columns=['cmdb_id','count'])
    unique_names = int(df[name_col].nunique()) if name_col in df.columns else 0
    name_counts = df[name_col].value_counts().head(20).reset_index().rename(columns={'index':name_col, name_col:'count'}) if name_col in df.columns else pd.DataFrame(columns=[name_col,'count'])
    overview = pd.DataFrame([{
        'file': file_label,
        'min_timestamp': ts_min,
        'max_timestamp': ts_max,
        'unique_cmdb_count': unique_cmdb,
        'unique_name_count': unique_names,
        'row_count': row_count
    }])
    return overview, cmdb_counts, name_counts

# Build overviews
metric_overview, metric_cmdb_counts, metric_kpi_counts = build_overview(metric_df, 'kpi_name', 'metric.csv')
trace_overview, trace_cmdb_counts, trace_trace_counts = build_overview(trace_df, 'trace_name', 'trace.csv')
log_overview, log_cmdb_counts, log_log_counts = build_overview(log_df, 'log_name', 'log.csv')

# For error logs: top 20 rows (selected columns) and total rows
error_row_count = len(error_df)
# Ensure selected columns exist
error_select_cols = [c for c in ['timestamp','cmdb_id','message'] if c in error_df.columns]
error_top20 = error_df[error_select_cols].head(20).copy()

# Put error overview as DataFrame too
error_overview = pd.DataFrame([{
    'file': 'error_logs.csv',
    'min_timestamp': int(error_df['timestamp'].min()) if 'timestamp' in error_df.columns and not error_df['timestamp'].isna().all() else None,
    'max_timestamp': int(error_df['timestamp'].max()) if 'timestamp' in error_df.columns and not error_df['timestamp'].isna().all() else None,
    'unique_cmdb_count': int(error_df['cmdb_id'].nunique()) if 'cmdb_id' in error_df.columns else 0,
    'unique_name_count': None,  # not applicable for error logs
    'row_count': error_row_count
}])

# Display compact summaries (limited to top 20 where applicable)
metric_overview, metric_cmdb_counts, metric_kpi_counts, trace_overview, trace_cmdb_counts, trace_trace_counts, log_overview, log_cmdb_counts, log_log_counts, error_overview, error_top20

```
Out[1]:
```
import pandas as pd

# Build a concise English summary using the previously loaded overview variables
summary = (
    "Summary of telemetry files:\n\n"
    "metric.csv:\n"
    f"- Time range: min_timestamp={int(metric_overview.at[0,'min_timestamp'])}, max_timestamp={int(metric_overview.at[0,'max_timestamp'])}\n"
    f"- Rows: {int(metric_overview.at[0,'row_count'])}; Unique cmdb_ids: {int(metric_overview.at[0,'unique_cmdb_count'])}; Unique KPI names: {int(metric_overview.at[0,'unique_name_count'])}\n"
    "- Top cmdb_id counts (up to 20):\n"
    f"{metric_cmdb_counts.head(20).to_string(index=False)}\n"
    "- Top KPI name counts (up to 20):\n"
    f"{metric_kpi_counts.head(20).to_string(index=False)}\n\n"
    "trace.csv:\n"
    f"- Time range: min_timestamp={int(trace_overview.at[0,'min_timestamp'])}, max_timestamp={int(trace_overview.at[0,'max_timestamp'])}\n"
    f"- Rows: {int(trace_overview.at[0,'row_count'])}; Unique cmdb_ids: {int(trace_overview.at[0,'unique_cmdb_count'])}; Unique trace names: {int(trace_overview.at[0,'unique_name_count'])}\n"
    "- Top cmdb_id counts (up to 20):\n"
    f"{trace_cmdb_counts.head(20).to_string(index=False)}\n"
    "- Top trace name counts (up to 20):\n"
    f"{trace_trace_counts.head(20).to_string(index=False)}\n\n"
    "log.csv:\n"
    f"- Time range: min_timestamp={int(log_overview.at[0,'min_timestamp'])}, max_timestamp={int(log_overview.at[0,'max_timestamp'])}\n"
    f"- Rows: {int(log_overview.at[0,'row_count'])}; Unique cmdb_ids: {int(log_overview.at[0,'unique_cmdb_count'])}; Unique log names: {int(log_overview.at[0,'unique_name_count'])}\n"
    "- Top cmdb_id counts (up to 20):\n"
    f"{log_cmdb_counts.head(20).to_string(index=False)}\n"
    "- Top log name counts (up to 20):\n"
    f"{log_log_counts.head(20).to_string(index=False)}\n\n"
    "error_logs.csv:\n"
    "- This file is empty: no timestamps, no cmdb_id values, total rows = 0.\n"
    "- Top rows (up to 20): none.\n\n"
    "Notes:\n"
    "- metric.csv contains the largest volume (29,554 rows) with 26 unique components and 340 KPI names.\n"
    "- trace.csv has 7,176 rows, 12 components and 72 trace features.\n"
    "- log.csv is small (527 rows) with 7 components and 3 log features.\n"
    "- No error log entries are present in error_logs.csv.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  min_timestamp  max_timestamp  unique_cmdb_count  unique_name_count  row_count
0  metric.csv     1616653800     1616655600                 26                340      29554,             count  count
0         Mysql01   4893
1         Mysql02   4884
2        Tomcat02   2016
3        Tomcat03   2007
4        Tomcat04   1994
5        Tomcat01   1919
6         Redis02   1768
7            MG02   1593
8            MG01   1560
9         Redis01   1488
10           IG02   1139
11           IG01   1102
12       apache02    902
13       apache01    891
14  ServiceTest10    124
15   ServiceTest1    124
16  ServiceTest11    124
17   ServiceTest2    124
18   ServiceTest4    124
19   ServiceTest8    124,                                                 count  count
0          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    350
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    349
2                  OSLinux-OSLinux_ZABBIX_Host_Uptime    349
3        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    347
4      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    346
5                         OSLinux-CPU_CPU_CPUidleutil    342
6                             OSLinux-CPU_CPU_CPULoad    342
7                          OSLinux-CPU_CPU_CPUSysTime    342
8                         OSLinux-CPU_CPU_CPUUserTime    342
9                              OSLinux-CPU_CPU_CPUWio    342
10  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    341
11                                             app.rr    340
12                                            app.mrt    340
13                                            app.cnt    340
14                                             app.sr    340
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    294
16                         OSLinux-CPU_CPU_CPUCpuUtil    286
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    286
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    273
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    272,         file  min_timestamp  max_timestamp  unique_cmdb_count  unique_name_count  row_count
0  trace.csv     1616653800     1616655600                 12                 72       7176,        count  count
0       MG02   1212
1       MG01   1212
2   Tomcat03    477
3   Tomcat02    477
4   dockerB1    477
5   dockerA2    477
6   Tomcat04    477
7       IG02    474
8       IG01    474
9   Tomcat01    474
10  dockerA1    474
11  dockerB2    471,                             count  count
0        trace.self.duration_mean    372
1         trace.self.duration_p95    372
2            trace.self.row_count    372
3     trace.to_MG01.duration_mean    247
4      trace.to_MG01.duration_p95    247
5         trace.to_MG01.row_count    247
6     trace.to_MG02.duration_mean    246
7         trace.to_MG02.row_count    246
8      trace.to_MG02.duration_p95    246
9    trace.from_IG02.duration_p95    124
10      trace.from_MG02.row_count    124
11  trace.from_IG01.duration_mean    124
12      trace.from_IG02.row_count    124
13   trace.from_IG01.duration_p95    124
14      trace.from_IG01.row_count    124
15  trace.from_MG02.duration_mean    124
16   trace.from_MG02.duration_p95    124
17  trace.from_IG02.duration_mean    124
18  trace.from_MG01.duration_mean    123
19   trace.from_MG01.duration_p95    123,       file  min_timestamp  max_timestamp  unique_cmdb_count  unique_name_count  row_count
0  log.csv     1616653800     1616655600                  7                  3        527,       count  count
0  Tomcat04     88
1  Tomcat02     85
2  Tomcat01     85
3  Tomcat03     85
4  apache01     62
5  apache02     62
6   Mysql01     60,               count  count
0   log.error_count    216
1     log.row_count    216
2  log.gc_pause_max     95,              file min_timestamp max_timestamp  unique_cmdb_count unique_name_count  row_count
0  error_logs.csv          None          None                  0              None          0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df already loaded in the session.
start_ts = 1616653800
end_ts = 1616655600
tz = pytz.timezone('Asia/Shanghai')

# Ensure expected columns exist
assert set(['timestamp','cmdb_id','kpi_name','value']).issubset(metric_df.columns), "metric_df missing required columns"

# 1) Global stats per (cmdb_id, kpi_name) computed on full series (before any window filtering)
grp_value = metric_df.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grp_value.agg(
    global_p95=lambda x: float(x.quantile(0.95)),
    global_p50=lambda x: float(x.quantile(0.50)),
    global_max=lambda x: float(x.max()),
    total_count='count'
).reset_index()

# 2) Windowed stats for the given incident window (inclusive)
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

if not window_df.empty:
    grp_win = window_df.groupby(['cmdb_id','kpi_name'])
    window_stats = grp_win.agg(
        count_in_window=('value','count'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max'),
        max_value_in_window=('value','max')
    ).reset_index()
else:
    # empty window -> create empty dataframe with expected columns
    window_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','earliest_timestamp_in_window','latest_timestamp_in_window','max_value_in_window'])

# 3) Merge global and window stats
merged = pd.merge(global_stats, window_stats, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric window counts with 0 and keep timestamps as NaN where no data
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
# earliest/latest timestamp: keep as Int64 nullable to allow NaN
merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].astype('Int64')
merged['latest_timestamp_in_window'] = merged['latest_timestamp_in_window'].astype('Int64')
# max_value_in_window: float, NaN where no data
merged['max_value_in_window'] = merged['max_value_in_window'].astype('float')

# Also compute localized datetime columns for internal use (per rule about timezone), not part of final required columns
# (these are kept as intermediate columns and won't be displayed in final compact table)
merged['earliest_ts_local'] = merged['earliest_timestamp_in_window'].apply(
    lambda t: pd.to_datetime(int(t), unit='s').tz_localize(pytz.UTC).tz_convert(tz) if not pd.isna(t) else pd.NaT
)
merged['latest_ts_local'] = merged['latest_timestamp_in_window'].apply(
    lambda t: pd.to_datetime(int(t), unit='s').tz_localize(pytz.UTC).tz_convert(tz) if not pd.isna(t) else pd.NaT
)

# Decide sorting: if all count_in_window are zero, sort by total_count desc, else by count_in_window desc
if merged['count_in_window'].sum() == 0:
    merged_sorted = merged.sort_values(by=['total_count','global_max'], ascending=[False,False])
else:
    merged_sorted = merged.sort_values(by=['count_in_window','total_count'], ascending=[False,False])

# Select required columns and limit to top 20 rows for compact output
result_cols = [
    'cmdb_id','kpi_name','global_p95','global_p50','global_max','total_count',
    'count_in_window','earliest_timestamp_in_window','latest_timestamp_in_window','max_value_in_window'
]
result_df_top20 = merged_sorted[result_cols].head(20).reset_index(drop=True)

# Display the compact table variable
result_df_top20

```
Out[2]:
```
summary = (
    "Summary of metric analysis (top 20 by count_in_window):\n\n"
    "- All top 20 (cmdb_id, kpi_name) pairs have full coverage inside the incident window: count_in_window = 31 (earliest=1616653800, latest=1616655600).\n"
    "- Most top entries are JVM memory and thread metrics for IG01, IG02, MG01, MG02 (HeapMemoryMax ~2.040136e+09; HeapMemoryUsed up to ~1.026358e+09; NoHeapMemoryUsed ~5.8e+07; ThreadCount ~70–77).\n"
    "- MG02 also shows several OS CPU metrics among the top 20 (CPULoad, CPUSysTime, CPUUserTime, CPUWio) with global_max values shown in the table.\n"
    "- Each listed pair has total_count = 31 (the same as count_in_window), indicating the global series for these KPIs only contains points in the incident window.\n"
    "- Output limited to 20 rows; no anomaly or threshold calculations were performed—this is a raw summary of counts and maxima within the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     kpi_name    global_p95    global_p50    global_max  total_count  count_in_window  earliest_timestamp_in_window  latest_timestamp_in_window  max_value_in_window
0     IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09           31               31                    1616653800                  1616655600         2.040136e+09
1     IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.192109e+08  4.035191e+08  8.476836e+08           31               31                    1616653800                  1616655600         8.476836e+08
2     IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07  5.846119e+07  5.846119e+07           31               31                    1616653800                  1616655600         5.846119e+07
3     IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  7.400000e+01  7.000000e+01  7.500000e+01           31               31                    1616653800                  1616655600         7.500000e+01
4     IG02     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09           31               31                    1616653800                  1616655600         2.040136e+09
5     IG02    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.669595e+08  5.131642e+08  8.965025e+08           31               31                    1616653800                  1616655600         8.965025e+08
6     IG02  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.860966e+07  5.860966e+07  5.860966e+07           31               31                    1616653800                  1616655600         5.860966e+07
7     IG02     JVM-Threads_7778_JVM_ThreadCount_Threads  7.700000e+01  7.300000e+01  7.700000e+01           31               31                    1616653800                  1616655600         7.700000e+01
8     MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09           31               31                    1616653800                  1616655600         2.040136e+09
9     MG01    JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.571785e+08  6.006775e+08  1.026358e+09           31               31                    1616653800                  1616655600         1.026358e+09
10    MG01  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.886139e+07  5.886139e+07  5.886139e+07           31               31                    1616653800                  1616655600         5.886139e+07
11    MG01     JVM-Threads_7779_JVM_ThreadCount_Threads  7.500000e+01  7.200000e+01  7.600000e+01           31               31                    1616653800                  1616655600         7.600000e+01
12    MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09           31               31                    1616653800                  1616655600         2.040136e+09
13    MG02    JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  8.985296e+08  5.603144e+08  9.482346e+08           31               31                    1616653800                  1616655600         9.482346e+08
14    MG02  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.902065e+07  5.899076e+07  5.902065e+07           31               31                    1616653800                  1616655600         5.902065e+07
15    MG02     JVM-Threads_7779_JVM_ThreadCount_Threads  7.400000e+01  7.200000e+01  7.400000e+01           31               31                    1616653800                  1616655600         7.400000e+01
16    MG02                      OSLinux-CPU_CPU_CPULoad  1.690000e+00  1.260000e+00  1.970000e+00           31               31                    1616653800                  1616655600         1.970000e+00
17    MG02                   OSLinux-CPU_CPU_CPUSysTime  1.200301e+00  1.073800e+00  1.294900e+00           31               31                    1616653800                  1616655600         1.294900e+00
18    MG02                  OSLinux-CPU_CPU_CPUUserTime  2.645155e+01  2.635191e+01  2.660920e+01           31               31                    1616653800                  1616655600         2.660920e+01
19    MG02                       OSLinux-CPU_CPU_CPUWio  5.574000e-01  9.151000e-02  4.399218e+00           31               31                    1616653800                  1616655600         4.399218e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables: metric_df, start_ts, end_ts, tz
# Ensure required vars exist; otherwise set them (defensive)
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'start_ts' not in globals():
    start_ts = 1616653800
if 'end_ts' not in globals():
    end_ts = 1616655600
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and total_count per (cmdb_id, kpi_name) using the full series
grp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
global_p95_df = grp.quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
total_count_df = grp.count().reset_index().rename(columns={'value':'total_count'})
global_stats = pd.merge(global_p95_df, total_count_df, on=['cmdb_id','kpi_name'])

# 2) Filter points to the incident window (inclusive)
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Within the window compute per-pair aggregates
if not window_df.empty:
    window_agg = window_df.groupby(['cmdb_id','kpi_name']).agg(
        count_in_window=('value','count'),
        max_value_in_window=('value','max'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max')
    ).reset_index()
else:
    window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','max_value_in_window','earliest_timestamp_in_window','latest_timestamp_in_window'])

# 4) Identify anomalies: points in window where value >= global_p95 (global threshold computed earlier)
# Merge global_p95 into window_df rows to evaluate per-row
if not window_df.empty:
    window_with_p95 = pd.merge(window_df, global_stats, on=['cmdb_id','kpi_name'], how='left')
    # Mark anomalies where value >= global_p95 (handle NaN global_p95 conservatively -> False)
    window_with_p95['is_anomaly'] = window_with_p95.apply(
        lambda r: False if pd.isna(r['global_p95']) else (r['value'] >= r['global_p95']),
        axis=1
    )
    anomalies = window_with_p95[window_with_p95['is_anomaly']].copy()
    if not anomalies.empty:
        anomalies_agg = anomalies.groupby(['cmdb_id','kpi_name']).agg(
            count_anomalies_in_window=('value','count'),
            earliest_anomaly_timestamp_in_window=('timestamp','min'),
            latest_anomaly_timestamp_in_window=('timestamp','max')
        ).reset_index()
    else:
        anomalies_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
else:
    anomalies_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])

# 5) Merge all stats together
merged = global_stats.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')
merged = merged.merge(anomalies_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric window counts with 0 and ensure types
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
merged['max_value_in_window'] = merged['max_value_in_window'].astype('float')
merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].astype('Int64')
merged['latest_timestamp_in_window'] = merged['latest_timestamp_in_window'].astype('Int64')

merged['count_anomalies_in_window'] = merged['count_anomalies_in_window'].fillna(0).astype(int)
merged['earliest_anomaly_timestamp_in_window'] = merged['earliest_anomaly_timestamp_in_window'].astype('Int64')
merged['latest_anomaly_timestamp_in_window'] = merged['latest_anomaly_timestamp_in_window'].astype('Int64')

# 6) Compute breach_percent = (max_value_in_window - global_p95) / global_p95; use NaN if global_p95 == 0 or max_value_in_window is NaN
def compute_breach(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp) or gp == 0 or pd.isna(mv):
        return np.nan
    return float((mv - gp) / gp)

merged['breach_percent'] = merged.apply(compute_breach, axis=1)

# 7) Prepare final anomaly table sorted by breach_percent desc then by count_anomalies_in_window desc
anomaly_candidates = merged[merged['count_anomalies_in_window'] > 0].copy()

if not anomaly_candidates.empty:
    anomaly_candidates_sorted = anomaly_candidates.sort_values(by=['breach_percent','count_anomalies_in_window'], ascending=[False,False])
    result_cols = [
        'cmdb_id','kpi_name','total_count','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
        'max_value_in_window','global_p95','breach_percent'
    ]
    result_df = anomaly_candidates_sorted[result_cols].head(20).reset_index(drop=True)
else:
    # No anomalies found: return top 20 by count_in_window with their global_p95 and max_value_in_window (plus total_count)
    fallback = merged.sort_values(by=['count_in_window','total_count'], ascending=[False,False])
    result_df = fallback[['cmdb_id','kpi_name','total_count','count_in_window','global_p95','max_value_in_window']].head(20).reset_index(drop=True)

# Final compact result
result_df

```
Out[3]:
```
Summary of detected high-value events (based on metric.csv, anomalies = points in the incident window >= that pair's global P95):

Key findings (top offenders by breach_percent):
- IG01 — OSLinux_NETWORK_TCP-FIN-WAIT: breach_percent = 19.0 (extreme relative spike). total_count=20, all 20 points in-window, 1 anomaly at 1616655180 (value 1.0 vs global_p95=0.05). Suggest investigating socket/TCP teardown/backlog on IG01.
- MG02 — OSLinux CPU WIO: breach_percent ≈ 6.89, total_count=31, count_in_window=31, 2 anomaly points (earliest 1616653860, latest 1616654700). High I/O wait on MG02.
- Tomcat04 — OSLinux CPU WIO: breach_percent ≈ 4.69, total_count=31, 2 anomalies (1616654520–1616654940). Indicates I/O wait spikes on Tomcat04.
- Tomcat03 — OSLinux CPU WIO: breach_percent ≈ 3.44, total_count=31, 2 anomalies (1616654280–1616654760).
- Several disk-related metrics (MG01, Tomcat04, Tomcat03, Tomcat02, Tomcat01) show breach_percent ≈ 2.33 (single anomaly each) for DSKRTps/DSKRead/DSKBps/DSKBps-like metrics — indicates transient elevated disk read/throughput/rtps.
- MG01 — OSLinux-CPU single-CPU idle anomaly (breach_percent ≈ 2.33) — large relative departure in single-CPU idle percentage at one timestamp.
- Mysql01 / Mysql02:
  - Mysql01 ThreadsRunning: total_count=31, count_in_window=31, count_anomalies_in_window=31, breach_percent = 1.0 (global_p95=1, max=2). Every in-window point >= global_p95 — sustained higher thread count.
  - Mysql02 Innodb data pending fsyncs: total_count=31, count_in_window=31, count_anomalies_in_window=20, breach_percent = 1.0 — notable sustained/infrequent pending fsyncs.

Other notes:
- Most top entries have full coverage inside the window (count_in_window = total_count), meaning the KPIs' data for these pairs is entirely within the incident window.
- Anomalies are mostly short-lived (1–2 timestamps) for OS/disk metrics, except MySQL ThreadsRunning which is elevated across the entire window (31/31).
- breach_percent is the relative exceedance of the in-window max over the global P95; very large breach_percent values can result from low baselines (e.g., TCP-FIN-WAIT baseline ≈0.05).

Suggested next steps:
- Immediately investigate IG01 network/socket state (TCP FIN/close behavior and upstream/downstream connections).
- Investigate high I/O wait (CPU WIO) on MG02 and Tomcat hosts and check disk subsystem health and queueing for MG01/Tomcat nodes where disk metrics spiked.
- Investigate MySQL: elevated ThreadsRunning and Innodb pending fsyncs — check query load, long transactions, fsync latency, and disk flush behavior.
- Correlate these metric timestamps with application logs and traces to identify root cause paths.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_count  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp_in_window  latest_anomaly_timestamp_in_window  max_value_in_window  global_p95  breach_percent
0       IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           20               20                          1                            1616655180                          1616655180             1.000000    0.050000       19.000000
1       MG02                             OSLinux-CPU_CPU_CPUWio           31               31                          2                            1616653860                          1616654700             4.399218    0.557400        6.892390
2   Tomcat04                             OSLinux-CPU_CPU_CPUWio           31               31                          2                            1616654520                          1616654940             2.333417    0.409925        4.692309
3   Tomcat03                             OSLinux-CPU_CPU_CPUWio           31               31                          2                            1616654280                          1616654760             2.792227    0.628450        3.443038
4       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps           15               15                          1                            1616654940                          1616654940             0.016900    0.005070        2.333333
5   Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps           15               15                          1                            1616654940                          1616654940             3.816700    1.145010        2.333333
6       MG01                    OSLinux-CPU_CPU-3_SingleCpuidle           15               15                          1                            1616653860                          1616653860            89.195938   26.758781        2.333333
7       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead           15               15                          1                            1616654940                          1616654940             0.135600    0.040680        2.333333
8   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...           15               15                          1                            1616654160                          1616654160             0.300000    0.090000        2.333333
9   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps           15               15                          1                            1616654340                          1616654340             2.000000    0.600000        2.333333
10  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead           15               15                          1                            1616654940                          1616654940           126.133300   37.839990        2.333333
11      MG01                             OSLinux-CPU_CPU_CPUWio           30               30                          2                            1616653800                          1616654460             1.754532    0.537165        2.266281
12  apache02                             OSLinux-CPU_CPU_CPUWio           19               19                          1                            1616653800                          1616653800             1.563400    0.526600        1.968857
13  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps           14               14                          1                            1616654340                          1616654340            13.483333    4.860022        1.774336
14  apache02                            OSLinux-CPU_CPU_CPULoad           19               19                          1                            1616653800                          1616653800             0.230000    0.086000        1.674419
15  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead           14               14                          1                            1616654340                          1616654340           765.466667  288.106688        1.656886
16      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps           15               15                          1                            1616654340                          1616654340             4.000000    1.900000        1.105263
17   Mysql01                    Mysql-MySQL_3306_ThreadsRunning           31               31                         31                            1616653800                          1616655600             2.000000    1.000000        1.000000
18   Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs           31               31                         20                            1616653800                          1616655600             2.000000    1.000000        1.000000
19   Mysql01               Mysql-MySQL_3306_Max trx rows locked           31               31                          2                            1616655060                          1616655420             1.000000    0.500000        1.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df and window timestamps if available, otherwise load/define
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if 'start_ts' not in globals():
    start_ts = 1616653800
if 'end_ts' not in globals():
    end_ts = 1616655600
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')

# Ensure required columns exist
assert set(['timestamp','cmdb_id','trace_name','value']).issubset(trace_df.columns), "trace_df missing required columns"

# 1) Global P95 and total_count per (cmdb_id, trace_name) using the full series
grp = trace_df.groupby(['cmdb_id','trace_name'])['value']
global_p95_df = grp.quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
total_count_df = grp.count().reset_index().rename(columns={'value':'total_count'})
global_stats = pd.merge(global_p95_df, total_count_df, on=['cmdb_id','trace_name'])

# 2) Filter points to the incident window (inclusive)
window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Within the window compute per-pair aggregates (count_in_window, max value, earliest/latest timestamps)
if not window_df.empty:
    window_agg = window_df.groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','count'),
        max_value_in_window=('value','max'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max')
    ).reset_index()
else:
    window_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','max_value_in_window','earliest_timestamp_in_window','latest_timestamp_in_window'])

# 4) Identify anomalies: rows in window where value >= global_p95 (global threshold computed earlier)
if not window_df.empty:
    window_with_p95 = pd.merge(window_df, global_stats, on=['cmdb_id','trace_name'], how='left')
    window_with_p95['is_anomaly'] = window_with_p95.apply(
        lambda r: False if pd.isna(r['global_p95']) else (r['value'] >= r['global_p95']),
        axis=1
    )
    anomalies = window_with_p95[window_with_p95['is_anomaly']].copy()
    if not anomalies.empty:
        anomalies_agg = anomalies.groupby(['cmdb_id','trace_name']).agg(
            count_anomalies_in_window=('value','count'),
            earliest_anomaly_timestamp_in_window=('timestamp','min'),
            latest_anomaly_timestamp_in_window=('timestamp','max')
        ).reset_index()
    else:
        anomalies_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
else:
    anomalies_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])

# 5) Merge global_stats, window_agg, anomalies_agg
merged = global_stats.merge(window_agg, on=['cmdb_id','trace_name'], how='left')
merged = merged.merge(anomalies_agg, on=['cmdb_id','trace_name'], how='left')

# 6) Fill and cast types for compact display
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
merged['max_value_in_window'] = merged['max_value_in_window'].astype('float')
merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].astype('Int64')
merged['latest_timestamp_in_window'] = merged['latest_timestamp_in_window'].astype('Int64')

merged['count_anomalies_in_window'] = merged['count_anomalies_in_window'].fillna(0).astype(int)
merged['earliest_anomaly_timestamp_in_window'] = merged['earliest_anomaly_timestamp_in_window'].astype('Int64')
merged['latest_anomaly_timestamp_in_window'] = merged['latest_anomaly_timestamp_in_window'].astype('Int64')

# 7) Compute breach_percent = (max_value_in_window - global_p95) / global_p95
def compute_breach(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp) or gp == 0 or pd.isna(mv):
        return np.nan
    return float((mv - gp) / gp)

merged['breach_percent'] = merged.apply(compute_breach, axis=1)

# 8) Prepare final table
anomaly_rows = merged[merged['count_anomalies_in_window'] > 0].copy()

if not anomaly_rows.empty:
    # Sort by earliest_anomaly_timestamp_in_window asc, then breach_percent desc
    anomaly_rows_sorted = anomaly_rows.sort_values(by=['earliest_anomaly_timestamp_in_window','breach_percent'], ascending=[True,False])
    result_cols = [
        'cmdb_id','trace_name','total_count','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
        'max_value_in_window','global_p95','breach_percent'
    ]
    result_df = anomaly_rows_sorted[result_cols].head(20).reset_index(drop=True)
else:
    # Fallback: no anomalies found -> top 20 by count_in_window with global_p95 and max_value_in_window
    fallback = merged.sort_values(by=['count_in_window','total_count'], ascending=[False,False])
    result_df = fallback[['cmdb_id','trace_name','total_count','count_in_window','global_p95','max_value_in_window']].head(20).reset_index(drop=True)

# Final compact output
result_df

```
Out[4]:
```
Summary of trace analysis (anomalies = in-window points >= that pair's global P95):

- Overall: multiple (cmdb_id, trace_name) pairs showed high-value trace events during the incident window. Most affected pairs are traces between MG01/MG02 and dockerA/dockerB nodes (both directions: trace.to_* and trace.from_*). Each listed pair has count_in_window = total_count and typically 2 anomaly points within the window.

- Timing & magnitude:
  - Earliest anomalies start at 1616653800. The table is sorted by earliest anomaly timestamp; many pairs show anomalies beginning at the window start.
  - Breach_percent (relative exceedance of the window max over global P95) ranges ~0.07 to ~0.45. The largest relative increase: MG01 trace.from_dockerB2.duration_mean (breach ≈ 0.452), and the next highest are MG02/docker pairs (breach ≈ 0.386 and 0.197 for duration_mean/p95).
  - Most trace duration anomalies are modest increases (10–45% above the usual 95th percentile). Row_count anomalies also appear (e.g., MG01/dockerA1 row_count elevated by ~10.9%).

- Notable entries:
  - MG01 <-> dockerB2 and MG02 <-> dockerA1 show repeated duration_mean anomalies (2 anomaly points each).
  - MG02 had duration_p95 anomalies (to/from dockerA1 and dockerB2).
  - Tomcat01 shows a small trace set (4 points) with an early p95/mean anomaly at 1616653800 (single-timestamp spike, breach ~0.16–0.17).
  - Many anomalies are symmetric (to_MG vs from_MG), suggesting cross-service latency degradation rather than one-sided instrumentation noise.

- Interpretation & suggested next steps:
  - These results point to increased trace latencies and some elevated request counts on MG01/MG02 <-> dockerA/dockerB interactions during the incident window.
  - Investigate network/path latency and service dependencies between MG and docker nodes, and correlate with host metrics (CPU, I/O, network) and application logs at the anomaly timestamps.
  - Pay attention to Tomcat01's early spike as a possible localized event to correlate with request traces or logs.

This is a compact summary of the top trace anomalies (limited to 20 rows) found within the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  total_count  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp_in_window  latest_anomaly_timestamp_in_window  max_value_in_window  global_p95  breach_percent
0       MG01  trace.from_dockerB2.duration_mean           31               31                          2                            1616653800                          1616654040             0.196038    0.134989        0.452246
1   dockerB2        trace.to_MG01.duration_mean           31               31                          2                            1616653800                          1616654040             0.196038    0.134989        0.452246
2       MG02  trace.from_dockerA1.duration_mean           30               30                          2                            1616653800                          1616654820             0.186327    0.134447        0.385876
3   dockerA1        trace.to_MG02.duration_mean           30               30                          2                            1616653800                          1616654820             0.186327    0.134447        0.385876
4       MG02   trace.from_dockerA1.duration_p95           30               30                          2                            1616653800                          1616655540             1.219700    1.018550        0.197487
5   dockerA1         trace.to_MG02.duration_p95           30               30                          2                            1616653800                          1616655540             1.219700    1.018550        0.197487
6       MG02  trace.from_dockerB1.duration_mean           31               31                          2                            1616653800                          1616654040             0.141647    0.119475        0.185584
7   dockerB1        trace.to_MG02.duration_mean           31               31                          2                            1616653800                          1616654040             0.141647    0.119475        0.185584
8   Tomcat01       trace.from_root.duration_p95            4                4                          1                            1616653800                          1616653800             0.832800    0.713497        0.167208
9   Tomcat01      trace.from_root.duration_mean            4                4                          1                            1616653800                          1616653800             0.208200    0.181887        0.144666
10      MG01      trace.from_dockerA1.row_count           31               31                          2                            1616653800                          1616654520            97.000000   87.500000        0.108571
11  dockerA1            trace.to_MG01.row_count           31               31                          2                            1616653800                          1616654520            97.000000   87.500000        0.108571
12      MG01  trace.from_dockerB1.duration_mean           31               31                          2                            1616653800                          1616654160             0.143077    0.129115        0.108132
13  dockerB1        trace.to_MG01.duration_mean           31               31                          2                            1616653800                          1616654160             0.143077    0.129115        0.108132
14      MG01   trace.from_dockerA2.duration_p95           31               31                          2                            1616653800                          1616654040             1.223500    1.126250        0.086349
15  dockerA2         trace.to_MG01.duration_p95           31               31                          2                            1616653800                          1616654040             1.223500    1.126250        0.086349
16      MG01      trace.from_dockerA2.row_count           31               31                          2                            1616653800                          1616654520            86.000000   79.500000        0.081761
17  dockerA2            trace.to_MG01.row_count           31               31                          2                            1616653800                          1616654520            86.000000   79.500000        0.081761
18      MG02   trace.from_dockerB2.duration_p95           30               30                          2                            1616653800                          1616654760             1.094950    1.018815        0.074729
19  dockerB2         trace.to_MG02.duration_p95           30               30                          2                            1616653800                          1616654760             1.094950    1.018815        0.074729```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables where possible
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")
if 'start_ts' not in globals():
    start_ts = 1616653800
if 'end_ts' not in globals():
    end_ts = 1616655600
if 'tz' not in globals():
    tz = pytz.timezone('Asia/Shanghai')

# Ensure required columns exist
assert set(['timestamp','cmdb_id','log_name','value']).issubset(log_df.columns), "log_df missing required columns"

# 1) Compute global P95 and total_count per (cmdb_id, log_name) using the full series (before filtering)
grp_full = log_df.groupby(['cmdb_id','log_name'])['value']
global_p95_df = grp_full.quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
total_count_df = grp_full.count().reset_index().rename(columns={'value':'total_count'})
global_stats = pd.merge(global_p95_df, total_count_df, on=['cmdb_id','log_name'])

# 2) Filter points to the incident window (inclusive)
window_df = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) Within the window compute per-pair aggregates
if not window_df.empty:
    window_agg = window_df.groupby(['cmdb_id','log_name']).agg(
        count_in_window=('value','count'),
        max_value_in_window=('value','max'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max')
    ).reset_index()
else:
    window_agg = pd.DataFrame(columns=['cmdb_id','log_name','count_in_window','max_value_in_window','earliest_timestamp_in_window','latest_timestamp_in_window'])

# 4) Identify anomalies: rows in window where value >= global_p95
if not window_df.empty:
    window_with_p95 = pd.merge(window_df, global_stats, on=['cmdb_id','log_name'], how='left')
    window_with_p95['is_anomaly'] = window_with_p95.apply(
        lambda r: False if pd.isna(r['global_p95']) else (r['value'] >= r['global_p95']),
        axis=1
    )
    anomalies = window_with_p95[window_with_p95['is_anomaly']].copy()
    if not anomalies.empty:
        anomalies_agg = anomalies.groupby(['cmdb_id','log_name']).agg(
            count_anomalies_in_window=('value','count'),
            earliest_anomaly_timestamp_in_window=('timestamp','min'),
            latest_anomaly_timestamp_in_window=('timestamp','max')
        ).reset_index()
    else:
        anomalies_agg = pd.DataFrame(columns=['cmdb_id','log_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
else:
    anomalies_agg = pd.DataFrame(columns=['cmdb_id','log_name','count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])

# 5) Merge all stats together
merged = global_stats.merge(window_agg, on=['cmdb_id','log_name'], how='left')
merged = merged.merge(anomalies_agg, on=['cmdb_id','log_name'], how='left')

# 6) Fill and cast types for compact display
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
merged['max_value_in_window'] = merged['max_value_in_window'].astype('float')
merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].astype('Int64')
merged['latest_timestamp_in_window'] = merged['latest_timestamp_in_window'].astype('Int64')

merged['count_anomalies_in_window'] = merged['count_anomalies_in_window'].fillna(0).astype(int)
merged['earliest_anomaly_timestamp_in_window'] = merged['earliest_anomaly_timestamp_in_window'].astype('Int64')
merged['latest_anomaly_timestamp_in_window'] = merged['latest_anomaly_timestamp_in_window'].astype('Int64')

# 7) Compute breach_percent = (max_value_in_window - global_p95) / global_p95
def compute_breach(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp) or gp == 0 or pd.isna(mv):
        return np.nan
    return float((mv - gp) / gp)

merged['breach_percent'] = merged.apply(compute_breach, axis=1)

# 8) Prepare final table sorted by earliest_anomaly_timestamp_in_window asc, then breach_percent desc
anomaly_rows = merged[merged['count_anomalies_in_window'] > 0].copy()

if not anomaly_rows.empty:
    anomaly_rows_sorted = anomaly_rows.sort_values(by=['earliest_anomaly_timestamp_in_window','breach_percent'], ascending=[True,False])
    result_cols = [
        'cmdb_id','log_name','total_count','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
        'max_value_in_window','global_p95','breach_percent'
    ]
    result_df = anomaly_rows_sorted[result_cols].head(20).reset_index(drop=True)
else:
    # Fallback: return top 20 by count_in_window with global_p95 and max_value_in_window
    fallback = merged.sort_values(by=['count_in_window','total_count'], ascending=[False,False])
    result_df = fallback[['cmdb_id','log_name','total_count','count_in_window','global_p95','max_value_in_window']].head(20).reset_index(drop=True)

# Display compact result
result_df

```
Out[5]:
```
Summary of log analysis (anomalies = in-window points >= that pair’s global P95):

Key points
- Many log.error_count series (Mysql01, Tomcat01–04, apache01–02) show count_in_window = total_count and count_anomalies_in_window = count_in_window, but global_p95 = 0 and max_value_in_window = 0. This is an artifact: global_p95==0 makes the >= check mark every point as an “anomaly” and breach_percent is NaN. These entries are not meaningful without handling zero baseline — investigate whether error_count is always zero or if P95 computation needs adjustment.
- GC pause metrics:
  - Tomcat02 log.gc_pause_max: total_count=23, count_in_window=23, 4 anomaly points, max_in_window=0.10, global_p95=0.08, breach_percent ≈ 0.25 — modest GC pause spikes.
  - Tomcat01 and Tomcat04/03 also show gc_pause_max anomalies (some breach_percent = 0.0 where max == global_p95).
- log.row_count spikes (small relative increases) across multiple hosts (Tomcat01–04, apache01/02, Mysql01):
  - Mysql01 log.row_count shows the largest relative increase: max_in_window=21.0 vs global_p95=16.3 → breach_percent ≈ 0.288 (notable).
  - Other row_count entries show smaller breaches (≈0.01–0.08).
- Timing: earliest anomaly timestamps appear around 1616653800–1616653980 for many series; several series have anomalies across most or all of the incident window.

Recommendations / next steps
- Treat the log.error_count results cautiously: re-evaluate P95 computation (or exclude zeros) so zeros don’t generate false positives. Confirm whether error counts are truly zero during the window.
- Investigate Tomcat GC pause spikes (Tomcat02 in particular) at the listed timestamps — check JVM GC logs and application throughput around those times.
- Investigate Mysql01 log.row_count increase — correlate with DB workload, queries or connection spikes.
- Correlate these log anomalies with metrics/traces (CPU, I/O, traces) and application logs at the anomaly timestamps to find root causes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  total_count  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp_in_window  latest_anomaly_timestamp_in_window  max_value_in_window  global_p95  breach_percent
0    Mysql01   log.error_count           30               30                         30                            1616653800                          1616655540                 0.00      0.0000             NaN
1   Tomcat01   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
2   Tomcat02   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
3   Tomcat03   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
4   Tomcat04   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
5   apache01   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
6   apache02   log.error_count           31               31                         31                            1616653800                          1616655600                 0.00      0.0000             NaN
7   Tomcat02  log.gc_pause_max           23               23                          4                            1616653920                          1616655180                 0.10      0.0800        0.250000
8   Tomcat01  log.gc_pause_max           23               23                          4                            1616653920                          1616655360                 0.09      0.0900        0.000000
9   Tomcat02     log.row_count           31               31                          2                            1616653980                          1616655420               583.00    555.5000        0.049505
10  apache02     log.row_count           31               31                          2                            1616653980                          1616654520              1114.00   1082.0000        0.029575
11  Tomcat04  log.gc_pause_max           26               26                          2                            1616653980                          1616655240                 0.10      0.0975        0.025641
12  Tomcat01     log.row_count           31               31                          2                            1616653980                          1616654460               573.00    565.5000        0.013263
13   Mysql01     log.row_count           30               30                          2                            1616654040                          1616654160                21.00     16.3000        0.288344
14  apache01     log.row_count           31               31                          2                            1616654460                          1616654520              1132.00   1072.5000        0.055478
15  Tomcat03     log.row_count           31               31                          2                            1616654520                          1616654820               575.00    532.0000        0.080827
16  Tomcat04     log.row_count           31               31                          2                            1616654520                          1616655480               562.00    541.5000        0.037858
17  Tomcat03  log.gc_pause_max           23               23                          6                            1616654580                          1616655420                 0.09      0.0900        0.000000```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse dataframes and window bounds from prior steps; if missing, load them
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")
if 'start_ts' not in globals():
    start_ts = 1616653800
if 'end_ts' not in globals():
    end_ts = 1616655600
tz = pytz.timezone('Asia/Shanghai')

# Components of interest
components = ['Mysql01','Mysql02','MG01','MG02','IG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# Helper to build per-series anomaly summary for a dataframe
def build_anomaly_summary(df, name_col, dtype):
    # global p95 and total_count per series (computed on full series)
    grp = df.groupby(['cmdb_id', name_col])['value']
    global_p95 = grp.quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    total_count = grp.count().reset_index().rename(columns={'value':'total_count'})
    stats = pd.merge(global_p95, total_count, on=['cmdb_id', name_col])
    
    # windowed aggregates
    window_df = df[(df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)].copy()
    if not window_df.empty:
        win_agg = window_df.groupby(['cmdb_id', name_col]).agg(
            count_in_window=('value','count'),
            max_value_in_window=('value','max'),
            earliest_timestamp_in_window=('timestamp','min'),
            latest_timestamp_in_window=('timestamp','max')
        ).reset_index()
    else:
        win_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'count_in_window','max_value_in_window','earliest_timestamp_in_window','latest_timestamp_in_window'])
    
    # identify anomalies (value >= global_p95) on window rows
    if not window_df.empty:
        window_with_p95 = pd.merge(window_df, stats, on=['cmdb_id', name_col], how='left')
        window_with_p95['is_anomaly'] = window_with_p95.apply(
            lambda r: False if pd.isna(r['global_p95']) else (r['value'] >= r['global_p95']),
            axis=1
        )
        anomalies = window_with_p95[window_with_p95['is_anomaly']].copy()
        if not anomalies.empty:
            anom_agg = anomalies.groupby(['cmdb_id', name_col]).agg(
                count_anomalies_in_window=('value','count'),
                earliest_anomaly_timestamp_in_window=('timestamp','min'),
                latest_anomaly_timestamp_in_window=('timestamp','max')
            ).reset_index()
        else:
            anom_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
    else:
        anom_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'count_anomalies_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
    
    # merge everything
    merged = stats.merge(win_agg, on=['cmdb_id', name_col], how='left')
    merged = merged.merge(anom_agg, on=['cmdb_id', name_col], how='left')
    
    # fill and types
    merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].astype('float')
    merged['earliest_timestamp_in_window'] = merged['earliest_timestamp_in_window'].astype('Int64')
    merged['latest_timestamp_in_window'] = merged['latest_timestamp_in_window'].astype('Int64')
    merged['count_anomalies_in_window'] = merged['count_anomalies_in_window'].fillna(0).astype(int)
    merged['earliest_anomaly_timestamp_in_window'] = merged['earliest_anomaly_timestamp_in_window'].astype('Int64')
    merged['latest_anomaly_timestamp_in_window'] = merged['latest_anomaly_timestamp_in_window'].astype('Int64')
    
    # breach percent
    def compute_breach(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp) or gp == 0 or pd.isna(mv):
            return np.nan
        return float((mv - gp) / gp)
    merged['breach_percent'] = merged.apply(compute_breach, axis=1)
    
    # add series name column unified and dtype
    merged = merged.rename(columns={name_col: 'series_name'})
    merged['type'] = dtype
    # keep only relevant columns
    keep_cols = [
        'cmdb_id','series_name','type','total_count','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
        'max_value_in_window','global_p95','breach_percent'
    ]
    # ensure all cols present
    for c in keep_cols:
        if c not in merged.columns:
            merged[c] = pd.NA
    return merged[keep_cols]

# Build summaries for metrics, traces, logs
metric_summary = build_anomaly_summary(metric_df, 'kpi_name', 'metric')
trace_summary = build_anomaly_summary(trace_df, 'trace_name', 'trace')
log_summary = build_anomaly_summary(log_df, 'log_name', 'log')

# Combine all summaries
all_summary = pd.concat([metric_summary, trace_summary, log_summary], ignore_index=True)

# Restrict to selected components
all_summary_sel = all_summary[all_summary['cmdb_id'].isin(components)].copy()

# For each component compute requested aggregations
agg_rows = []
for cm in components:
    comp_df = all_summary_sel[all_summary_sel['cmdb_id'] == cm].copy()
    # series that have >=1 anomaly in-window
    has_anom = comp_df[comp_df['count_anomalies_in_window'] > 0].copy()
    number_of_anomaly_series = int(has_anom['series_name'].nunique())
    total_anomaly_points = int(has_anom['count_anomalies_in_window'].sum())
    # earliest anomaly timestamp across all series (may be NaN)
    if not has_anom.empty:
        earliest_anomaly_timestamp = int(has_anom['earliest_anomaly_timestamp_in_window'].min())
    else:
        earliest_anomaly_timestamp = pd.NA
    # max breach percent ignoring NaN
    if not has_anom.empty and has_anom['breach_percent'].dropna().size > 0:
        max_breach_percent = float(has_anom['breach_percent'].dropna().max())
        # dominant series = series_name (type) that has this max breach (take first if ties)
        dominant_row = has_anom.loc[has_anom['breach_percent'].fillna(-np.inf).idxmax()]
        dominant_series = f"{dominant_row['series_name']} ({dominant_row['type']})"
    else:
        max_breach_percent = np.nan
        dominant_series = pd.NA
    # sustained flag: any series with count_anomalies_in_window >= 50% of its count_in_window
    sustained = False
    for _, r in has_anom.iterrows():
        ciw = int(r['count_in_window'])
        if ciw > 0 and r['count_anomalies_in_window'] >= 0.5 * ciw:
            sustained = True
            break
    sustained_flags = 'sustained' if sustained else ''
    agg_rows.append({
        'cmdb_id': cm,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'number_of_anomaly_series': number_of_anomaly_series,
        'total_anomaly_points': total_anomaly_points,
        'max_breach_percent': max_breach_percent,
        'dominant_series': dominant_series,
        'sustained_flags': sustained_flags
    })

consolidated_df = pd.DataFrame(agg_rows)

# Sort by earliest_anomaly_timestamp ascending (NaN/NA go last)
consolidated_df['earliest_sort'] = consolidated_df['earliest_anomaly_timestamp'].apply(lambda x: np.iinfo(np.int64).max if pd.isna(x) else int(x))
consolidated_df = consolidated_df.sort_values(by=['earliest_sort'], ascending=True).drop(columns=['earliest_sort']).reset_index(drop=True)

# Keep final columns and limit to top 20 rows
final_cols = ['cmdb_id','earliest_anomaly_timestamp','number_of_anomaly_series','total_anomaly_points','max_breach_percent','dominant_series','sustained_flags']
consolidated_df = consolidated_df[final_cols].head(20)

# Display compact consolidated candidate table
consolidated_df

```
Out[6]:
```
summary = (
    "Consolidated anomaly candidates (components restricted):\n\n"
    "Overview:\n"
    "- All listed components (Mysql01, Mysql02, MG01, MG02, IG01, Tomcat01–04) show anomalies beginning at the incident window start (earliest_anomaly_timestamp = 1616653800).\n"
    "- Every component has at least one series with sustained anomalies (sustained flag set) meaning one or more series had anomalies covering >=50% of their in-window points.\n\n"
    "Highlights & priorities:\n"
    "- IG01: highest relative spike (max_breach_percent = 19.0). Dominant series: OSLinux_NETWORK_TCP-FIN-WAIT (metric). Investigate networking/socket teardown issues first for IG01.\n"
    "- MG02: pronounced I/O-wait issue (max_breach_percent ≈ 6.89). Dominant series: OSLinux CPU WIO (metric). Prioritize disk/IO wait and related processes on MG02.\n"
    "- Tomcat04 and Tomcat03: notable CPU WIO spikes (max_breach_percent ≈ 4.69 and 3.44). Dominant series: OSLinux CPU WIO (metric) — check disk/IO and thread scheduling on these app nodes.\n"
    "- MG01, Tomcat01, Tomcat02: mid-level disk/IO symptoms (max_breach_percent ≈ 2.33). Dominant series are disk RTps/DSK Bps metrics — inspect disk throughput and latency.\n"
    "- Mysql01 and Mysql02: sustained database signals (max_breach_percent = 1.0) and very large counts of anomaly series/points (Mysql01 total_anomaly_points=2708, Mysql02=2482). Dominant series include MySQL threads/Innodb pending fsyncs. Treat DB sustained load and fsync/backpressure as high priority.\n\n"
    "Interpretation:\n"
    "- The incident appears to begin at the window start across components, with both transient high breaches (IG01 network spike) and sustained issues (MySQL metrics and multiple disk/IO-related metrics across MG and Tomcat nodes).\n"
    "- Immediate investigation order: 1) IG01 network/TCP state, 2) MG02 (I/O wait) and Tomcat hosts (I/O wait & disk metrics), 3) Mysql01/Mysql02 sustained DB anomalies (threads, pending fsyncs).\n\n"
    "Next steps:\n"
    "- Correlate these component-level anomalies with logs and traces at the listed timestamps, check disk subsystems, network connectivity and MySQL slow queries/flush behavior.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  earliest_anomaly_timestamp  number_of_anomaly_series  total_anomaly_points  max_breach_percent                                    dominant_series sustained_flags
0   Mysql01                  1616653800                       227                  2708            1.000000      Mysql-MySQL_3306_Max trx rows locked (metric)       sustained
1   Mysql02                  1616653800                       224                  2482            1.000000  Mysql-MySQL_3306_Innodb data pending fsyncs (m...       sustained
2      MG01                  1616653800                       148                   871            2.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTp...       sustained
3      MG02                  1616653800                       146                   882            6.892390                    OSLinux-CPU_CPU_CPUWio (metric)       sustained
4      IG01                  1616653800                       106                   583           19.000000  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (...       sustained
5  Tomcat01                  1616653800                       139                  1267            2.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...       sustained
6  Tomcat02                  1616653800                       143                  1338            2.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...       sustained
7  Tomcat03                  1616653800                       142                  1304            3.443038                    OSLinux-CPU_CPU_CPUWio (metric)       sustained
8  Tomcat04                  1616653800                       142                  1301            4.692309                    OSLinux-CPU_CPU_CPUWio (metric)       sustained```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse previously loaded DataFrames and window bounds
# metric_df, trace_df, start_ts, end_ts should exist from prior steps
tz = pytz.timezone('Asia/Shanghai')

# Define target series
targets = [
    {'type':'metric','cmdb_id':'Mysql01','series_name':'Mysql-MySQL_3306_ThreadsRunning','name_col':'kpi_name'},
    {'type':'metric','cmdb_id':'Mysql02','series_name':'Mysql-MySQL_3306_Innodb data pending fsyncs','name_col':'kpi_name'},
    {'type':'trace','cmdb_id':'MG01','series_name':'trace.from_dockerB2.duration_mean','name_col':'trace_name'},
    {'type':'trace','cmdb_id':'MG02','series_name':'trace.from_dockerA1.duration_mean','name_col':'trace_name'},
    {'type':'metric','cmdb_id':'IG01','series_name':'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT','name_col':'kpi_name'},
    {'type':'metric','cmdb_id':'MG02','series_name':'OSLinux-CPU_CPU_CPUWio','name_col':'kpi_name'}
]

# Helper to get global_p95 for a series using the full series (rule 9)
def compute_global_p95(df, cmdb_id, name_col, series_name):
    sel = df[(df['cmdb_id'] == cmdb_id) & (df[name_col] == series_name)]
    if sel.empty:
        return np.nan
    return float(sel['value'].quantile(0.95))

# Collect per-series limited raw points and summaries
rows_list = []   # for combined limited raw points
summary_rows = []

for t in targets:
    typ = t['type']
    cm = t['cmdb_id']
    sname = t['series_name']
    name_col = t['name_col']
    source_df = metric_df if typ == 'metric' else trace_df
    
    # Compute global_p95 on full series before window filtering
    gp95 = compute_global_p95(source_df, cm, name_col, sname)
    
    # Filter to the series (full) then to window (inclusive)
    series_full = source_df[(source_df['cmdb_id'] == cm) & (source_df[name_col] == sname)].copy()
    series_win = series_full[(series_full['timestamp'] >= start_ts) & (series_full['timestamp'] <= end_ts)].copy()
    
    # Add global_p95 and is_anomaly
    series_win = series_win[['timestamp','value']].copy()
    series_win['global_p95'] = gp95
    # treat NaN global_p95 as not anomaly
    series_win['is_anomaly'] = series_win['value'].apply(lambda v: False if pd.isna(gp95) else (v >= gp95))
    
    # Limit to first 20 rows per series (preserve chronological order)
    series_win_limited = series_win.sort_values(by='timestamp').head(20).copy()
    series_win_limited['series_label'] = f"{cm} | {sname} ({typ})"
    # Reorder columns
    series_win_limited = series_win_limited[['series_label','timestamp','value','global_p95','is_anomaly']]
    
    # Append to combined list
    rows_list.append(series_win_limited)
    
    # Compute per-series summary
    total_rows_in_window = int(len(series_win))
    count_anomalies = int(series_win['is_anomaly'].sum()) if total_rows_in_window>0 else 0
    if count_anomalies > 0:
        earliest_anom_ts = int(series_win.loc[series_win['is_anomaly'], 'timestamp'].min())
        latest_anom_ts = int(series_win.loc[series_win['is_anomaly'], 'timestamp'].max())
    else:
        earliest_anom_ts = pd.NA
        latest_anom_ts = pd.NA
    max_value_in_window = float(series_win['value'].max()) if total_rows_in_window>0 else np.nan
    
    summary_rows.append({
        'series_label': f"{cm} | {sname} ({typ})",
        'total_rows_in_window': total_rows_in_window,
        'count_anomalies_in_window': count_anomalies,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'latest_anomaly_timestamp': latest_anom_ts,
        'max_value_in_window': max_value_in_window,
        'global_p95': gp95
    })

# Combine limited raw points into one DataFrame (up to 20 rows per series)
if rows_list:
    combined_points_df = pd.concat(rows_list, ignore_index=True)
else:
    combined_points_df = pd.DataFrame(columns=['series_label','timestamp','value','global_p95','is_anomaly'])

# Build summary DataFrame (one-line per series)
summary_df = pd.DataFrame(summary_rows, columns=[
    'series_label','total_rows_in_window','count_anomalies_in_window',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','global_p95'
])

# Final compact outputs: combined_points_df (first 20 rows per series) and summary_df (one row per series)
combined_points_df, summary_df

```
Out[7]:
```
Summary of requested in-window raw points and precomputed global P95 (one-line per series):

- Mysql01 — Mysql-MySQL_3306_ThreadsRunning (metric): global_p95=1.0. In-window rows=31, anomalies=31, earliest_anomaly=1616653800, latest_anomaly=1616655600, max_in_window=2.0. (All in-window points >= global_p95.)

- Mysql02 — Mysql-MySQL_3306_Innodb data pending fsyncs (metric): global_p95=1.0. In-window rows=31, anomalies=20, earliest_anomaly=1616653800, latest_anomaly=1616655600, max_in_window=2.0.

- MG01 — trace.from_dockerB2.duration_mean (trace): global_p95≈0.134989. In-window rows=31, anomalies=2, earliest_anomaly=1616653800, latest_anomaly=1616654040, max_in_window≈0.196038.

- MG02 — trace.from_dockerA1.duration_mean (trace): global_p95≈0.134447. In-window rows=30, anomalies=2, earliest_anomaly=1616653800, latest_anomaly=1616654820, max_in_window≈0.186327.

- IG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (metric): global_p95=0.05. In-window rows=20, anomalies=1, earliest_anomaly=1616655180, latest_anomaly=1616655180, max_in_window=1.0.

- MG02 — OSLinux-CPU_CPU_CPUWio (metric): global_p95≈0.5574. In-window rows=31, anomalies=2, earliest_anomaly=1616653860, latest_anomaly=1616654700, max_in_window≈4.399218.

For each series, up to the first 20 in-window raw points (timestamp, value) were returned and include the series' global_p95 and an is_anomaly boolean (True when value >= global_p95). If you want the full per-series point lists expanded or timestamps converted to human-readable times, tell me which series to expand.

The original code execution output of IPython Kernel is also provided below for reference:

(                                          series_label   timestamp     value  global_p95  is_anomaly
0    Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...  1616653800  1.000000      1.0000        True
1    Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...  1616653860  1.000000      1.0000        True
2    Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...  1616653920  1.000000      1.0000        True
3    Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...  1616653980  1.000000      1.0000        True
4    Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...  1616654040  1.000000      1.0000        True
..                                                 ...         ...       ...         ...         ...
115             MG02 | OSLinux-CPU_CPU_CPUWio (metric)  1616654700  4.399218      0.5574        True
116             MG02 | OSLinux-CPU_CPU_CPUWio (metric)  1616654760  0.070674      0.5574       False
117             MG02 | OSLinux-CPU_CPU_CPUWio (metric)  1616654820  0.224700      0.5574       False
118             MG02 | OSLinux-CPU_CPU_CPUWio (metric)  1616654880  0.104000      0.5574       False
119             MG02 | OSLinux-CPU_CPU_CPUWio (metric)  1616654940  0.091510      0.5574       False

[120 rows x 5 columns],                                         series_label  total_rows_in_window  count_anomalies_in_window  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  global_p95
0  Mysql01 | Mysql-MySQL_3306_ThreadsRunning (met...                    31                         31                  1616653800                1616655600             2.000000    1.000000
1  Mysql02 | Mysql-MySQL_3306_Innodb data pending...                    31                         20                  1616653800                1616655600             2.000000    1.000000
2   MG01 | trace.from_dockerB2.duration_mean (trace)                    31                          2                  1616653800                1616654040             0.196038    0.134989
3   MG02 | trace.from_dockerA1.duration_mean (trace)                    30                          2                  1616653800                1616654820             0.186327    0.134447
4  IG01 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN...                    20                          1                  1616655180                1616655180             1.000000    0.050000
5             MG02 | OSLinux-CPU_CPU_CPUWio (metric)                    31                          2                  1616653860                1616654700             4.399218    0.557400)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```